<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_heatsink_new.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.2 — A Pulse of Power on a Finned Heat Sink

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L8.2 · Transient Heat**

The heat sink of Ex_08.1 is at the air temperature when its module is switched
on at 150 W for twenty seconds - a motor started, a drive in overload - and
then off. How hot does the module get, how fast does the heat spread into the
base, and how long until the sink has recovered? It is the heat equation: the
steady problem with one more term, the heat the aluminium stores. The
transient has an **exact solution** too, so a time-stepping grid method and a
physics-informed network are scored against the truth. Then the same network
finds, from one thermocouple glued to the base, the one number of a heat sink
that nobody knows to better than twenty per cent: the fins' heat transfer
coefficient.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics: two time scales, and the exact transient | — |
| **4** | Crank–Nicolson on a grid, refined until an agreed 0.05 K | — |
| **5** | a PINN with the lumped response built in, and the comparison | the trial function, the residual |
| **6** | the inverse problem: $h_{\mathrm{eff}}$ from one thermocouple | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About twenty minutes on a CPU, most of it the two trainings. Numbers are
printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'heatsink.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.2-transient-heat/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "heatsink"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "heatsink.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import heatsink as hs
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex08.2_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def online(fn, reps=5):                                    # the same, with the GPU made to finish before the clock stops
    def run():
        fn()
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    return middle(run, reps)

---

## 1 · The problem

The heat sink of Ex_08.1 - base plate 100 × 10 mm, ten fins of 2 × 40 mm, forced
air at 40 °C - sits at the air temperature. At $t$ = 0 its module is switched
on at 150 W; at $t$ = 20 s it is switched off. A thermocouple is glued to the
module face of the base plate, 80 mm from the left end, 15 mm beside the
module. Wanted: the temperature of the base plate at every point and instant
over 150 s, the hottest point under the module through the pulse, and the
thermocouple's record.

## 2 · The data

| | |
|---|---|
| heat sink, module, air | as in Ex_08.1: $k$ = 167 W/(m·K), $h_{\mathrm{eff}}$ = 322 W/(m²K) on the fin side, 50 kW/m² on the footprint |
| storage | $\rho c_p$ = 2.43 MJ/(m³K); diffusivity $\alpha = k/\rho c_p$ = 6.87e-05 m²/s |
| the pulse | 150 W from 0 to 20 s, then nothing; the sink at 40 °C at $t$ = 0 |
| window | 0 to 150 s |
| thermocouple | on the module face at $x$ = 80 mm; a reading every 2 s, noise 0.05 K |

## 3 · The physics

**Storage is the new term** (L8.2's first section): the heat that arrives and
is not conducted away raises the temperature,

$$\rho c_p\,T_t = k\,(T_{xx} + T_{zz})$$

with the same three walls as before - the module's flux, now switched on and
off, the convection on the fin side, the insulated ends - and one more
condition, the field at $t$ = 0. The equation is **parabolic**: it has a
direction in time and needs one initial field.

**Three time scales**, and only two matter. Heat crosses the 10 mm thickness
in $t_b^2/\alpha$ = 1.5 s, so the base plate is uniform through its thickness
almost at once. It spreads along the 100 mm base in $W^2/(\pi^2\alpha)$ =
15 s: that is how long the hot spot under the module takes to form, and to
level out after the power goes off. And the whole plate warms and cools with
the **lumped time constant** $\rho c_p t_b / h_{\mathrm{eff}}$ = 75 s, the heat
it stores over the rate the fins remove it. A 20 s pulse is short against that:
the sink reaches only a quarter of its steady rise, and takes the rest of the
window to give it back.

**The hand calculation in time** is the lumped model: one temperature for the
whole base, rising as $1 - e^{-t/\tau}$ while the power is on and decaying
with the same $\tau$ after. It is exact for the mean of a plate this thin and
misses, as before, the spreading.

**Scaled units.** $X$, $Z$ and $\theta$ as in Ex_08.1, $\tau = t/t_{\mathrm{end}}$
with $t_{\mathrm{end}}$ = 150 s:

$$\theta_\tau = \mathrm{Fo}\,(\varepsilon^2\theta_{XX} + \theta_{ZZ}), \qquad
\mathrm{Fo} = \frac{\alpha\,t_{\mathrm{end}}}{t_b^2} = 103, \qquad
\theta_Z = \mathrm{Bi}\,g(X)\,s(\tau)\ \text{at}\ Z = 1, \qquad \theta = 0\ \text{at}\ \tau = 0$$

with $s(\tau)$ = 1 while the module is on and 0 after, and the fin side and
the ends as before. (In L8.2 $\tau$ is a time constant; in this notebook it is
only the scaled time.)

**The exact solution.** Each cosine of the steady series has its own history.
What has not yet arrived is expanded in the eigenfunctions of the convecting
face, $\cos\mu_j(t_b - z)$ with $\mu_j \tan(\mu_j t_b) = h_{\mathrm{eff}}/k$ -
the transcendental equation of Incropera ch. 5, solved to machine precision -
and every term dies as $e^{-\alpha(\lambda_n^2 + \mu_j^2)t}$:

$$\theta(x, z, t) = \theta_{\mathrm{steady}}(x, z) - \sum_{n, j} c_{nj}\cos\lambda_n x\,\cos\mu_j(t_b - z)\,e^{-\alpha(\lambda_n^2 + \mu_j^2)t}$$

A pulse is two steps, the second subtracted from the first after 20 s
(`hs.exact_transient`). It is exact to the number of terms kept, and for any
$t$ above a fraction of a second a hundred terms are far more than enough. The
cell prints the data, draws the heat sink with its thermocouple, and plots the
exact hottest point, the thermocouple's reading and the field at the end of the
pulse.

In [ ]:
hs.describe_problem()
print(f"  in time    : through the thickness t_b²/α = {hs.T_BASE ** 2 / hs.ALPHA:.1f} s, along the base W²/(π²α) = {hs.W_BASE ** 2 / (np.pi ** 2 * hs.ALPHA):.1f} s, "
      f"lumped time constant {hs.TAU_LUMPED:.1f} s; the pulse lasts {hs.T_ON:.0f} s, the window {hs.T_END:.0f} s; Fo = {hs.FO:.0f}")
fig, ax = plt.subplots(figsize=(9.5, 4.6))
hs.draw_heatsink(ax, show_sensor=True)
plt.tight_layout(); plt.show()

t_plot = np.arange(0.5, hs.T_END + 1e-9, 0.5)
centre = np.array([hs.exact_transient(np.array([hs.W_BASE / 2]), np.array([hs.T_BASE]), t)[0] for t in t_plot])   # the module's centre
sensor = np.array([hs.exact_transient(np.array([hs.SENSOR]), np.array([hs.T_BASE]), t)[0] for t in t_plot])        # the thermocouple, exact
lumped = hs.THETA * ((1 - np.exp(-t_plot / hs.TAU_LUMPED)) - np.where(t_plot > hs.T_ON, 1 - np.exp(-(t_plot - hs.T_ON) / hs.TAU_LUMPED), 0))
print(f"hottest point: {hs.T_AIR + centre.max():.2f} °C at {t_plot[centre.argmax()]:.1f} s; the lumped model's peak {hs.T_AIR + lumped.max():.2f} °C; "
      f"the thermocouple peaks at {hs.T_AIR + sensor.max():.2f} °C at {t_plot[sensor.argmax()]:.1f} s, after the power is off; "
      f"at {hs.T_END:.0f} s the sink is {centre[-1]:.2f} K above the air")
xg, zg = np.linspace(0, hs.W_BASE, 401), np.linspace(0, hs.T_BASE, 41); XG, ZG = np.meshgrid(xg, zg)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw=dict(width_ratios=[1.1, 1]))
ax[0].plot(t_plot, hs.T_AIR + centre, lw=2, label="module centre, exact")
ax[0].plot(t_plot, hs.T_AIR + sensor, lw=2, label="thermocouple, exact")
ax[0].plot(t_plot, hs.T_AIR + lumped, "k--", lw=1, label="lumped model: one temperature")
ax[0].axvspan(0, hs.T_ON, color="tab:red", alpha=0.08); ax[0].text(hs.T_ON / 2, hs.T_AIR + 1, "power on", ha="center", fontsize=8, color="tab:red")
ax[0].set_xlabel("t  [s]"); ax[0].set_ylabel("temperature  [°C]"); ax[0].legend(frameon=False, fontsize=8)
c = ax[1].pcolormesh(XG * 1e3, ZG * 1e3, hs.T_AIR + hs.exact_transient(XG, ZG, hs.T_ON), cmap="inferno", shading="auto")
ax[1].set_aspect("equal"); ax[1].set_xlabel("x  [mm]"); ax[1].set_ylabel("z  [mm]"); ax[1].set_title("the base plate at 20 s, when the power goes off", fontsize=10)
plt.colorbar(c, ax=ax[1], label="°C"); plt.tight_layout(); plt.show()

**What you should see.** The module's centre climbing to 56.19 °C at 20 s, a
rise of 16.2 K where the lumped model says 10.8: the spreading is a third of
the rise while the power is on. When the power goes off the hot spot levels
out in about fifteen seconds and the whole plate then cools together, exactly
on the lumped curve. The thermocouple, 15 mm beside the module, **peaks four
seconds after the power is off**, at 50.0 °C: the heat under the module is
still arriving. At 150 s the sink is 2 K above the air.

---

## 4 · Finite differences in time, scored against the exact solution

**The same grid, stepped in time.** The five-point stencil and the ghost
nodes of Ex_08.1 give the space operator; **Crank–Nicolson** takes half of it
at the old time and half at the new, which is second order in time and stable
for any step. One factorisation of the matrix serves every step; the module's
flux is switched off at the step nearest 20 s (`hs.fdm_transient`).

**The agreed accuracy is 0.05 K**, the thermocouple's noise, at six recorded
instants: 7.5, 10, 20, 30, 60 and 150 s. The cell refines grid and step
together until it is met, scored against the exact field at every recorded
instant, and times one run.

In [ ]:
TARGET = 0.05                                                 # K: the agreed accuracy
REC_T = np.array([7.5, 10.0, 20.0, 30.0, 60.0, 150.0])        # the recorded instants, s
REC = REC_T / hs.T_END
exact_at = {}
for nx, nz, nt in ((41, 9, 150), (81, 17, 300), (161, 33, 600), (321, 65, 1200)):
    X, Z, TH = hs.fdm_transient(nx, nz, nt, REC)
    XX, ZZ = np.meshgrid(X * hs.W_BASE, Z * hs.T_BASE)
    errs = []
    for i, t in enumerate(REC_T):
        key = (nx, nz, t)
        exact_at[key] = hs.exact_transient(XX, ZZ, t)
        errs.append(np.abs(hs.kelvin(TH[i]) - exact_at[key]).max())
    sec = online(lambda: hs.fdm_transient(nx, nz, nt, REC), reps=3)
    print(f"  {nx:3d} x {nz:3d} nodes, {nt:5d} steps: worst error per instant", np.array2string(np.array(errs), precision=3),
          f"K, one run {sec:.2f} s" + ("   <- meets 0.05 K" if max(errs) <= TARGET else ""))
NX, NZ, NT = 321, 65, 1200                                    # the coarsest that meets it

**What you should see.** The worst error halving with every halving of the
spacing and the step - from about 0.4 K on the coarsest grid to 0.04 K on
321 × 65 nodes and 1200 steps - and largest just after the switch-on and the
switch-off, where the flux jumps and Crank–Nicolson is only first order. The
grid that meets 0.05 K has 20 865 unknowns and takes a few seconds for the
whole window: the transient costs the grid a thousand times the steady
problem, because it is solved once per step.

---

## 5 · A physics-informed network, and the comparison

**Time is one more input** (L8.2's *Space–Time Sampling*): the network takes
$(X, Z, \tau)$ and the whole block of space and time is trained at once, with
half of the points in the first fifth of the window, where the field changes
fastest.

**Build in what the hand calculation knows.** The lumped model
$\theta_L(\tau)$ - the exponential rise and fall with $\tau_{\mathrm{lumped}}$
= 75 s - is a solution of the equation for the mean of the plate, and it is
zero at $\tau$ = 0. So the trial function is

$$\hat\theta(X, Z, \tau) = \theta_L(\tau) + S\,\tau\,\mathcal{N}\big(\cos\pi X,\ Z,\ \tau\big)$$

The factor $\tau$ makes the start exact for any weights - **the hard initial
condition** of L8.2, the first remedy for causality - and $\cos\pi X$ makes
the ends insulated, as in Ex_08.1. The network learns the spreading, which is
up to a third of the rise, so $S$ = 0.2. Its clock is $\sqrt{\tau}$ rather
than $\tau$: heat spreads as the square root of time, so the first seconds,
where the field changes fastest, are stretched out for the network. The two
faces stay loss terms, the module's flux now multiplied by $s(\tau)$, both
scaled by Bi; the residual is scaled by Bi as well. **The energy balance in
time is a third loss term**: at every instant the fin-side mean of the field
must equal the lumped model, because the lumped model is exactly the heat
balance of the whole plate. It holds the level of the field through the
window, as its steady form did in Ex_08.1.

**TODO 1** is the trial function; **TODO 2** the residual of the scaled heat
equation, $(\hat\theta_\tau/\mathrm{Fo} - \varepsilon^2\hat\theta_{XX} - \hat\theta_{ZZ})/\mathrm{Bi}$.

In [ ]:
SCALE = 0.2                                                   # the size of the spreading over the lumped rise
TAU_ON = hs.T_ON / hs.T_END                                   # the switch-off, in scaled time
RATE = hs.T_END / hs.TAU_LUMPED                               # t_end over the lumped time constant

def lumped(tau, rate=RATE):
    # the lumped base plate in units of THETA: rises as 1 - exp(-t/tau_L) while on, falls with the same tau_L after
    on = 1.0 - torch.exp(-rate * tau)
    off = torch.where(tau > TAU_ON, 1.0 - torch.exp(-rate * torch.clamp(tau - TAU_ON, min=0.0)), torch.zeros_like(tau))
    return on - off

class BaseInTime(torch.nn.Module):                            # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=3, n_hidden=32, n_layers=4)       # dense tanh network: cos(pi X), Z, sqrt(tau) in
    def rate(self):
        return RATE                                           # the lumped rate; the inverse problem of section 6 makes it trainable
    def forward(self, p):
        X, Z, tau = p[:, 0:1], p[:, 1:2], p[:, 2:3]
        # TODO 1: the trial function - the lumped model at this rate, plus SCALE times tau times the network
        # evaluated on the features (cos(pi X), Z, sqrt(tau + 1e-12)), joined with torch.cat along dimension 1.
        #
        #   features = torch.cat([torch.cos(np.pi * X), Z, torch.sqrt(tau + 1e-12)], 1)
        #   return lumped(tau, self.rate()) + SCALE * tau * self.net(features)
        raise NotImplementedError("Write the trial function")

In [ ]:
# TODO 2: the residual of the scaled heat equation, (theta_tau / Fo - EPS^2 theta_XX - theta_ZZ) / Bi.
#
#   def residual(model, p):
#       th = model(p)
#       th_tau = grad(th, p)[:, 2:3]                              # column 2 is tau
#       return (th_tau / hs.FO - hs.EPS ** 2 * d2(th, p, 0) - d2(th, p, 1)) / hs.BI
raise NotImplementedError("Write residual(model, p)")

In [ ]:
W_F, W_E = 10.0, 10.0                                         # the weights of the two faces and of the energy balance
pts = hs.sample_base_time(4000)                               # (X, Z, tau), half of them in the first fifth of the window
top, bottom, left, right = hs.sample_faces_time(300, 60, 0)   # the faces, at random instants
P_in = to_tensor(pts, requires_grad=True)                     # collocation points, differentiable
P_top, P_bot = to_tensor(top, requires_grad=True), to_tensor(bottom, requires_grad=True)   # the two faces, differentiable
g_top = to_tensor((hs.footprint(top[:, 0]) * hs.pulse(top[:, 2]))[:, None])   # the module's flux, on while the power is on

tau_b, x_b = np.linspace(0, 1, 31), np.linspace(0, 1, 41)   # where the fin-side mean is taken: 41 points at 31 instants
P_bal = to_tensor(np.c_[np.tile(x_b, len(tau_b)), np.zeros(len(x_b) * len(tau_b)), np.repeat(tau_b, len(x_b))])   # (X, 0, tau) as a tensor
TAU_B = to_tensor(tau_b[:, None])                             # the same instants, for the lumped model

def face_loss(model, bi=hs.BI):
    flux = grad(model(P_top), P_top)[:, 1:2] / hs.BI - g_top             # the module face: theta_Z = Bi g s(tau)
    conv = grad(model(P_bot), P_bot)[:, 1:2] / hs.BI - (bi / hs.BI) * model(P_bot)   # the fin side: theta_Z = Bi theta
    return flux.pow(2).mean() + conv.pow(2).mean()

def balance_loss(model):
    mean_fin_side = model(P_bal).reshape(len(tau_b), len(x_b)).mean(1, keepdim=True)   # the fin-side mean at each instant
    return (mean_fin_side - lumped(TAU_B, model.rate())).pow(2).mean()   # equals the lumped model: the heat balance of the plate

model = BaseInTime().to(DEVICE)                               # four layers of 32
describe(model.net, n_collocation=len(P_in))                  # the parameter count and the collocation-to-parameter ratio
loss_fn = lambda: residual(model, P_in).pow(2).mean() + W_F * face_loss(model) + W_E * balance_loss(model)   # the composite loss
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
history = train_two_stage(model, loss_fn, adam_steps=1000, lbfgs_steps=300, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s (offline); residual {residual(model, P_in).pow(2).mean().item():.2e}, "
      f"faces {face_loss(model).item():.2e}, balance {balance_loss(model).item():.2e}")

In [ ]:
X, Z, TH = hs.fdm_transient(NX, NZ, NT, REC)                  # Crank-Nicolson on the grid that meets 0.05 K
XX, ZZ = np.meshgrid(X, Z)
T_true = np.array([exact_at[(NX, NZ, t)] for t in REC_T])     # the exact rise on the same nodes, at the recorded instants
def pinn_fields():
    with torch.no_grad():  # no gradient tracking: evaluation only
        return np.array([hs.kelvin(to_numpy(model(to_tensor(np.c_[XX.ravel(), ZZ.ravel(), np.full(XX.size, r)]))).reshape(XX.shape)) for r in REC])   # the network on the grid at each instant
T_f, T_p = hs.kelvin(TH), pinn_fields()
e_f = np.abs(T_f - T_true).reshape(len(REC), -1).max(axis=1)  # worst error at each instant
e_p = np.abs(T_p - T_true).reshape(len(REC), -1).max(axis=1)
t_fdm = online(lambda: hs.fdm_transient(NX, NZ, NT, REC), reps=3)   # one run over the window, on this runtime
t_pin = online(pinn_fields)                                   # the trained network at the six instants on the same nodes, on this runtime
i20 = list(REC_T).index(20.0)

# the time column is what runs ONLINE: a grid run over the window against an evaluation of the trained network at the
# recorded instants, on the same runtime. The training is offline, paid once, and reported on its own line.
print(f"{'':34s}{'unknowns':>10s}{'hottest at 20 s':>17s}{'worst error, any instant':>26s}{'time online':>14s}")
print(f"  {'exact series':32s}{'-':>10s}{hs.T_AIR + T_true[i20].max():14.2f} °C{'0 (the truth)':>26s}{'-':>14s}")
print(f"  {'Crank-Nicolson, ' + str(NX) + ' x ' + str(NZ) + ' x ' + str(NT):32s}{NX * NZ:10d}{hs.T_AIR + T_f[i20].max():14.2f} °C{e_f.max():23.2e} K{t_fdm:12.2f} s")
print(f"  {'PINN, 4 x 32':32s}{parameter_count(model):10d}{hs.T_AIR + T_p[i20].max():14.2f} °C{e_p.max():23.2e} K{t_pin:12.3f} s   (trained offline in {train_time:.0f} s, once)")
print("worst error per instant (7.5, 10, 20, 30, 60, 150 s):  grid", np.array2string(e_f, precision=3), " network", np.array2string(e_p, precision=3))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(t_plot, hs.T_AIR + centre, "k", lw=2, label="exact")
axes[0].plot(REC_T, hs.T_AIR + T_f.reshape(len(REC), -1).max(axis=1), "s", ms=7, label="Crank-Nicolson")
axes[0].plot(REC_T, hs.T_AIR + T_p.reshape(len(REC), -1).max(axis=1), "o", ms=5, label="PINN")
axes[0].set_xlabel("t  [s]"); axes[0].set_ylabel("hottest point  [°C]"); axes[0].legend(frameon=False)
axes[1].semilogy(REC_T, e_f, "s-", label=f"Crank-Nicolson, {NX} x {NZ} x {NT}")
axes[1].semilogy(REC_T, e_p, "o-", label="PINN")
axes[1].axhline(TARGET, color="k", lw=0.8, ls="--", label="agreed 0.05 K")
axes[1].set_xlabel("t  [s]"); axes[1].set_ylabel("worst error over the plate  [K]"); axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(cc.output_path("Ex08.2_report.png"), dpi=150); plt.show()

**What you should see.** The grid is within 0.04 K of the exact field at
every recorded instant and puts the hottest point at 56.18 °C at 20 s, against
the exact 56.19. The network does not meet the agreed accuracy: it is within
0.06 K at 150 s and 0.1 K at 60 s, but off by up to 1.3 K while the power is
on and at 20 s, where it puts the hottest point 1.1 K low. Its error sits at
the footprint's edges in the first half-minute, where the spreading front is
a few millimetres wide and moves fast: features finer in space and time than
four layers of 32 follow in this training. After the switch-off the field
smooths and the network catches up. (Its numbers vary a little from run to
run; the grid's do not.)

**By accuracy** the order is exact, grid, network, and only the grid meets
0.05 K throughout. **By time online** the network answers the six instants in
a few hundredths of a second where the grid needs three seconds for its 1200
steps - and the network could be read at any instant, not only the recorded
ones, without re-running anything. Its training is paid once, offline. What
would bring its early times in is L8.2's third remedy, **time windows**: a
network for the first thirty seconds alone, and another from there, which is
mini project MP8.2A's method.

---

## 6 · The inverse problem: the fins' coefficient from one thermocouple

The one number in this model that no handbook gives to better than twenty per
cent is $h_{\mathrm{eff}}$: it depends on the fan, the duct, the fin surface
and the fouling. Suppose it is **not known**, but the thermocouple recorded
the pulse, every 2 s with 0.05 K of noise. Can the same network find it?

It is L8.2's **Inverse Diffusivity** slide with a different unknown. One
number is trained beside the weights: $\mathrm{Bi} = \mathrm{Bi}_0 e^{\beta}$,
an exponential so that it stays positive. It is **not an output of the
network**: it is one more number the optimiser may change, and it enters the
loss in three places - the convection term on the fin side, the lumped
model built into the trial function, whose time constant is $\rho c_p
t_b/h_{\mathrm{eff}}$, and the energy balance that holds the field to it. The loss gains one term, the misfit to the readings.
$\beta$ starts at a coefficient 40 % too low. Nothing else changes.

**What identifies it.** While the power is on, the reading rises at a rate set
by the heat capacity, which is known; after the power is off, the reading
falls at a rate set by $h_{\mathrm{eff}}$ alone. The cooling tail is where the
information is, which is why the window runs to 150 s and not to 30. (Finding
the heat capacity as well, from the same record, is mini project MP8.2B.)

In [ ]:
t_s, r_s = hs.sensor_readings()                               # 75 readings: the rise above the air at the thermocouple, with noise
S_pts = to_tensor(np.c_[np.full_like(t_s, hs.SENSOR / hs.W_BASE), np.ones_like(t_s), t_s / hs.T_END])   # where and when they were read
S_val = to_tensor((r_s / hs.THETA)[:, None])                  # the readings, scaled

class Inverse(BaseInTime):                                    # the same network and trial function, and one more trainable number
    def __init__(self):
        super().__init__()                                    # a fresh network
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(0.6))))   # Bi = Bi_0 exp(beta), starts 40 % too low
    def bi(self):
        return hs.BI * torch.exp(self.beta)                   # the coefficient being identified, always positive
    def rate(self):
        return RATE * torch.exp(self.beta)                    # the lumped time constant moves with the unknown h_eff

inv = Inverse().to(DEVICE)                                    # the network and beta, on the device
W_D = 1000.0                                                  # the weight of the readings
def inverse_loss():
    data = (inv(S_pts) - S_val).pow(2).mean()                 # the misfit at the thermocouple
    return residual(inv, P_in).pow(2).mean() + W_F * face_loss(inv, inv.bi()) + W_E * balance_loss(inv) + W_D * data
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(inv, inverse_loss, adam_steps=1000, lbfgs_steps=300, lr=2e-3, report_every=0)
inv_time = time.perf_counter() - t0
h_found = float(inv.bi().detach()) / hs.BI * hs.H_EFF         # the identified number, back to W/(m²K)
print(f"h_eff: found {h_found:.1f} W/(m²K), true {hs.H_EFF:.1f} W/(m²K), off by {100 * (h_found / hs.H_EFF - 1):+.2f} %   ({inv_time:.0f} s)")
print(f"the lumped time constant that implies: {hs.RHO_CP * hs.T_BASE / h_found:.1f} s, true {hs.TAU_LUMPED:.1f} s")
with torch.no_grad():  # no gradient tracking: evaluation only
    fit = hs.kelvin(to_numpy(inv(S_pts)).ravel())
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(t_s, hs.T_AIR + r_s, ".", ms=4, label="thermocouple readings")
ax.plot(t_s, hs.T_AIR + fit, lw=1.5, label=f"the network's fit, h_eff = {h_found:.0f} W/(m²K)")
ax.axvspan(0, hs.T_ON, color="tab:red", alpha=0.08); ax.set_xlabel("t  [s]"); ax.set_ylabel("°C"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

**What you should see.** The coefficient found about 5 % low - 306 against
the true 322 W/(m²K) - from one thermocouple and a start that was 40 % off,
in some minutes on a CPU; the fit runs through the readings to their noise.
The time constant that goes with it, 79 s against 75, is what the readings
really measured. Five per cent is within what any handbook gives for
$h_{\mathrm{eff}}$, and the error has the same origin as the forward
network's: the first half-minute, where the network's field under the module
is low, is also where the thermocouple's rise is fitted.

---

## 7 · What the notebook says

* **The grid meets the agreed 0.05 K throughout; the network only after the
  pulse.** On what an engineer wants - the module reaches 56 °C at the end of
  a 20 s pulse, 5 K above what the lumped model says, and the sink has given
  most of it back after two time constants - the grid is exact to a hundredth
  and the network a kelvin low at the peak.
* **The spreading is a transient of its own.** The hot spot forms and
  disappears in about fifteen seconds, the plate cools in seventy-five. A
  thermocouple beside the module peaks after the power is off.
* **The exact solution kept the comparison honest.** Both methods were scored
  against the truth at six instants, and both are worst where the flux jumps.
* **Online, the network is the faster of the two**, because a time-stepping
  grid pays for every step and the network pays for none. It would be the one
  place in Ex_08 where the training is repaid on the forward problem alone,
  if it met the accuracy; in this training it does not in the first
  half-minute, and the honest reading is that the grid wins the forward
  transient.
* **The inverse problem is what the network is for.** The same code, with one
  more trainable number and one more loss term, returned $h_{\mathrm{eff}}$
  from one surface thermocouple. The grid solver would have to be run again
  and again inside an optimiser; the network carries the unknown in its loss
  and in the lumped model built into it.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex08.2_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L8.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "What does the storage term add to the steady problem of Ex_08.1, and what did the three time scales - through the thickness, along the base, the lumped time constant - tell you before anything was solved? (-> L8.2 Q1)": """
""",
    "The trial function is exactly zero at the start and carries the lumped model. Which of the lecture's remedies for causality is that, and where in time was the network's error largest? (-> L8.2 Q2)": """
""",
    "How was the transient checked, what made Crank-Nicolson first order at two instants, and why does the thermocouple peak after the power is off? (-> L8.2 Q3)": """
""",
    "In the inverse problem, what was trained besides the weights, in which places does it enter the loss, and which part of the record identified it? (-> L8.2 Q4)": """
""",
    "To conclude: the grid gave the whole window in seconds. For which task in this notebook did the network do something the grid solver could not do directly? (-> L8.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | unknowns | hottest at 20 s | worst error, any instant | time online |", "|---|---|---|---|---|",
            f"| exact series | - | {hs.T_AIR + T_true[i20].max():.2f} °C | 0 (the truth) | - |",
            f"| Crank-Nicolson, {NX} x {NZ} x {NT} | {NX * NZ} | {hs.T_AIR + T_f[i20].max():.2f} °C | {e_f.max():.2e} K | {t_fdm:.2f} s |",
            f"| PINN, 4 x 32 | {parameter_count(model)} | {hs.T_AIR + T_p[i20].max():.2f} °C | {e_p.max():.2e} K | {t_pin:.3f} s (trained offline in {train_time:.0f} s) |",
            "", f"Inverse problem: h_eff found {h_found:.1f} W/(m²K), true {hs.H_EFF:.1f} W/(m²K), off by {100 * (h_found / hs.H_EFF - 1):+.2f} %."]
    out = ["# Ex_08.2 — A Pulse of Power on a Finned Heat Sink", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex08.2_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex08.2_report.md into Ex08.2_report.pdf, with any figure
# saved as Ex08.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex08.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex08.2_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex08.2_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex08.2_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP8.2A · Thermal cycling

**The problem.** A drive does not pulse once. Switch the module on for 20 s and off for 40 s,
ten times. What swing of temperature does the module see once the cycles
repeat, and how many cycles until they do? That swing is what ages a power
module's solder.

| | |
|---|---|
| heat sink, module, air | as in this notebook |
| the train | 150 W for 20 s, nothing for 40 s, from $t$ = 0, ten cycles: 600 s |

**Why a plain PINN is not enough.** Ten pulses in one window: the field has ten kinks in time, and a network
trained over 600 s at once resolves none of them well. Train it over one period
with the field at the end of the previous period as its start, or give it the
cycle number as an input.

**What you build.** A PINN for $\theta(X, Z, \tau)$ with the lumped train built in (a sum of the
exponentials of this notebook) and the spreading learnt; the start of each period
taken from the end of the last.

**The ground truth you get.** `MP8.2A_new_truth.npz`, from `tools/miniprojects/heatsink_truth.py`: the exact
series by superposition of twenty steps, the module's centre every second.

**Worked example.** 

| cycle | peak | trough | swing |
|---|---|---|---|
| 1 | 16.19 K | 2.88 K | 13.31 K |
| 2 | 21.15 K | 9.25 K | 11.90 K |
| 5 | 24.88 K | 11.58 K | 13.30 K |
| 10 | 25.26 K | 11.80 K | 13.45 K |

The cycles repeat after about five; the module then swings by 13.5 K every minute about a mean 18 K above the air.

**Requirements.**

* The peak and the trough of every cycle within **0.3 K**.
* The number of cycles to a repeating swing, to the cycle.
* The field at the end of the tenth pulse within **0.5 K** everywhere.

**What you hand in.** The module's centre over 600 s, network against exact; the swing per cycle; and a paragraph on how the period was handled.

### MP8.2B · Heat capacity and coefficient together

**The problem.** Section 6 found $h_{\mathrm{eff}}$ with $\rho c_p$ known. Suppose neither is known:
the sink is of an alloy whose heat capacity is uncertain, in a duct whose
coefficient is unknown. Find both from the same thermocouple record, and show
which part of the record fixes which.

| | |
|---|---|
| record | the thermocouple of this notebook: 75 readings every 2 s, noise 0.05 K |
| unknowns | $\rho c_p$ (true 2.43 MJ/(m³K)) and $h_{\mathrm{eff}}$ (true 322 W/(m²K)); start both 30 % off |

**Why a plain PINN is not enough.** Two numbers that enter the same time constant $\rho c_p t_b / h_{\mathrm{eff}}$ can
trade against each other: the cooling tail fixes only their ratio. It is the rise
while the power is on, set by $\rho c_p$ alone, that separates them, and a network
that fits the tail well and the rise badly has found neither.

**What you build.** The inverse network of section 6 with two trainable numbers, both as exponentials,
entering the residual (through Fo), the fin-side term and the lumped model; then the
sensitivity of the fit to each, from the rise and from the tail separately.

**The ground truth you get.** `MP8.2B_new_readings.npz` to fit and `MP8.2B_new_truth.npz` to score against,
from `tools/miniprojects/heatsink_truth.py`: the record and the two true values.

**Worked example.** With both unknown, the rise over the first 20 s determines $\rho c_p$ to a few per cent on its own; with it fixed, the tail from 20 to 150 s determines $h_{\mathrm{eff}}$ as in section 6.

**Requirements.**

* Both numbers within **5 %** of the true values.
* A plot of the misfit against each unknown, with the other at its found value, showing the rise and the tail separately.
* One paragraph on which readings fix which number.

**What you hand in.** The two numbers found, the two misfit plots, and the paragraph.